# VERSE — E5 sur Google Colab

**Avant de commencer**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4** (ou mieux si disponible).
2. Dans votre Google Drive, créez le dossier `MyDrive/verse/` et déposez-y **`verse-egc2027.zip`**.
3. Exécutez les cellules **dans l'ordre**. Tous les résultats sont écrits **dans le Drive** : une déconnexion ne fait rien perdre.

**Après une déconnexion** : ré-exécutez les cellules 1, 2, 3 et 4 (environ 10 min), puis la cellule 6 : chaque processus reprend là où il s'était arrêté.

## 1. Drive et code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/verse'
assert os.path.exists(f'{D}/verse-egc2027.zip'), f"ZIP INTROUVABLE : déposez verse-egc2027.zip dans {D} (contenu actuel : {os.listdir('/content/drive/MyDrive')[:20]})"
os.makedirs(f'{D}/results', exist_ok=True); os.makedirs(f'{D}/generated', exist_ok=True)
!rm -rf /content/verse-egc2027 && unzip -q {D}/verse-egc2027.zip -d /content
%cd /content/verse-egc2027
# résultats et tableaux directement dans le Drive (sans écraser ce qui y est déjà)
!cp -rn results/. {D}/results/ && cp -rn paper/generated/. {D}/generated/
!rm -rf results paper/generated && ln -s {D}/results results && ln -s {D}/generated paper/generated
!ls results/ paper/generated/ | head -20
print('CELLULE 1 OK, dossier :', os.getcwd())
!pip -q install -r requirements.txt
!python3 -m unittest discover -s tests 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Installer et démarrer Ollama

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, os, time
env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_KEEP_ALIVE='-1', OLLAMA_MAX_LOADED_MODELS='1')
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True, env=env)
time.sleep(8)
!curl -s http://localhost:11434/api/version

## 3. Modèles (environ 5 min)
Fenêtre de contexte : 16k jetons pour Qwen (le texte intégral envoie jusqu'à 8 000 jetons), 8k pour le juge.

In [ ]:
!ollama pull qwen2.5:7b-instruct > /dev/null 2>&1
!ollama pull llama3.1:8b > /dev/null 2>&1
!printf 'FROM qwen2.5:7b-instruct\nPARAMETER num_ctx 16384\n' > /content/Mf.qwen && ollama create qwen16k -f /content/Mf.qwen
!printf 'FROM llama3.1:8b\nPARAMETER num_ctx 8192\n' > /content/Mf.llama && ollama create llama8k -f /content/Mf.llama
!ollama list

## 4. Vérification rapide (2 min)
Lancez cette cellule **à chaque reprise** : elle recharge le modèle en mémoire.

In [ ]:
%cd /content/verse-egc2027
!python3 experiments/run_llm.py --dry-run --n-patients 1 --out /content/dry.jsonl | tail -1
!python3 experiments/run_llm.py --model qwen16k --n-patients 1 --conds OURS,TEXT-FULL --out /content/essai.jsonl | tail -2
import json
for l in list(open('/content/essai.jsonl'))[:6]:
    x = json.loads(l); print(f"{x['cond']:10s} {x['latency']:5.1f}s  {x['question'][:45]}\n           -> {x['answer']}")

## 5. Contrôle avant le grand lancement
Dans la sortie ci-dessus, vérifiez que :
- les réponses sont **en français** et **sensées** ;
- la latence est raisonnable (environ 1 à 6 s par réponse sur T4).

Si une réponse de `TEXT-FULL` semble ignorer le début du contexte, la fenêtre de contexte n'est pas prise en compte : dans ce cas, arrêtez-vous et envoyez-moi la sortie.

## 6. Lancer E5 (4 processus en arrière-plan)
Les 8 conditions sont réparties en 4 lots équilibrés. **Pour reprendre après une déconnexion, ré-exécutez simplement cette cellule.**

In [ ]:
%cd /content/verse-egc2027
LOTS = {'A': 'TEXT-FULL,OURS', 'B': 'TEXT-RAG,OURS-GSonly', 'C': 'KG-DG,OURS-noHERSE', 'D': 'KG-PPR,OURS-noUP'}
!pkill -f run_llm.py; sleep 1
for k, c in LOTS.items():
    get_ipython().system(f'nohup python3 experiments/run_llm.py --model qwen16k --n-patients 50 --conds {c} '
                         f'--out results/e5_{k}.jsonl > results/log_{k}.txt 2>&1 &')
print('lancé :', LOTS)

## 7. Suivi (laisser tourner : garde aussi la session active)
Le total attendu est d'environ 6 000 réponses : 50 patients × 16 questions au plus × 8 conditions. Interrompez la cellule quand tout est fini ; ça n'arrête pas les calculs.

In [ ]:
%cd /content/verse-egc2027
import time, glob, subprocess, json
t0 = time.time(); n0 = None
while True:
    n = sum(sum(1 for _ in open(f)) for f in glob.glob('results/e5_*.jsonl'))
    n0 = n if n0 is None else n0
    alive = subprocess.run('pgrep -f run_llm.py', shell=True, capture_output=True).stdout.strip() != b''
    rate = (n - n0) / max(1, time.time() - t0) * 3600
    print(f"{time.strftime('%H:%M')}  {n} réponses  ~{rate:.0f}/h  " + ('en cours' if alive else 'TERMINÉ'), flush=True)
    if not alive:
        !tail -n 2 results/log_*.txt
        break
    time.sleep(600)

## 8. Fusion et score d'E5

In [ ]:
%cd /content/verse-egc2027
!cat results/e5_A.jsonl results/e5_B.jsonl results/e5_C.jsonl results/e5_D.jsonl > results/llm_qwen16k.jsonl
!python3 -c "import json;r=[json.loads(l) for l in open('results/llm_qwen16k.jsonl')];print(len(r),'réponses,',sum(x['answer'] is None for x in r),'échecs')"
!python3 experiments/score_llm.py results/llm_qwen16k.jsonl

## 9. Feuille d'annotation (200 réponses)
La feuille est écrite dans le Drive : `MyDrive/verse/results/annotation_qwen16k.csv`.

**Annotez-la dans Google Sheets** : ouvrez le fichier, remplissez la colonne `humain` (OK / KO), puis **Fichier → Télécharger → CSV**. Redéposez le fichier dans `MyDrive/verse/results/` **sous le même nom**. Excel en français remplace les virgules par des points-virgules : ne l'utilisez pas.

In [ ]:
%cd /content/verse-egc2027
!python3 experiments/make_annotation.py results/llm_qwen16k.jsonl --n 200

## 10. LLM juge sur les 200 réponses annotées (environ 5 min)
Cette étape peut tourner avant ou pendant l'annotation.

In [ ]:
%cd /content/verse-egc2027
import csv, json
keys = {(r['pid'], r['qid'], r['cond']) for r in csv.DictReader(open('results/annotation_qwen16k.csv', encoding='utf-8'))}
rows = [l for l in open('results/llm_qwen16k.jsonl') if (lambda x: (x['pid'], x['qid'], x['cond']))(json.loads(l)) in keys]
open('results/llm_qwen16k_ann200.jsonl', 'w').writelines(rows); print(len(rows), 'réponses à juger')
!python3 experiments/judge_llm.py results/llm_qwen16k_ann200.jsonl --model llama8k --out results/llm_qwen16k.judge_llama8k.jsonl | tail -2

## 11. Après l'annotation : vérifieurs contre l'humain, chiffres de l'article

In [ ]:
%cd /content/verse-egc2027
!python3 experiments/score_verifiers.py results/annotation_qwen16k.csv results/llm_qwen16k.judge_llama8k.jsonl
!python3 scripts/abstract_numbers.py --model qwen16k

## 12. Récupérer les résultats
Tout est déjà dans `MyDrive/verse/` (`results/` et `generated/`). Envoyez-moi `generated/summary_llm.json`, `generated/summary_verif_humain.json`, `generated/tab_llm_stats.tex` et la sortie de la cellule 11 : je rédige le paragraphe E5.